In [1]:
import pandas as pd
import time, sys, os
os.environ['SPS_HOME'] = '/beegfs/car/shenoy/SED-fitting/fsps/'
import fsps
import sedpy
import prospect
import emcee
import dynesty
import h5py
import numpy as np
import scipy
from matplotlib.pyplot import *
import matplotlib.colors as mcolors
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from matplotlib.patches import Circle
import glob
import os
from multiprocessing import Pool
from prospect.likelihood import lnlike_spec, lnlike_phot, write_log
from prospect.likelihood import chi_spec, chi_phot
from prospect.fitting import lnprobfn
from prospect.fitting import fit_model
from prospect.models import priors
from prospect.models.templates import TemplateLibrary
import matplotlib.pyplot as plt
# checking versions for compatibility
# vers = (np.__version__, scipy.__version__, h5py.__version__, fsps.__version__, prospect.__version__)
# print("numpy: {}\nscipy: {}\nh5py: {}\nfsps: {}\nprospect: {}".format(*vers))
# sedpy.__file__

df = pd.read_csv("flux_wide_wavelets_reg1_v2.csv")
print(df.columns)

flux_cols = [c for c in df.columns if c.startswith("F") and not c.endswith("_err") and not c.endswith("_n_eff")]
err_cols  = [c for c in df.columns if c.endswith("_n_eff")]
flux_cols = sorted(flux_cols)
err_cols  = sorted(err_cols)
df["fluxes"] = df[flux_cols].values.tolist()
df["errors"]  = df[err_cols].values.tolist()
filters_goods = ['F090W', 'F115W', 'F150W', 'F182M', 'F200W',
           'F210M', 'F277W', 'F335M', 'F356W', 'F410M', 'F430M', 'F444W', 'F480M']

Index(['region', 'n_pix', 'F090W', 'F090W_err', 'F115W', 'F115W_err', 'F150W',
       'F150W_err', 'F182M', 'F182M_err', 'F200W', 'F200W_err', 'F210M',
       'F210M_err', 'F277W', 'F277W_err', 'F335M', 'F335M_err', 'F356W',
       'F356W_err', 'F410M', 'F410M_err', 'F430M', 'F430M_err', 'F444W',
       'F444W_err', 'F480M', 'F480M_err'],
      dtype='object')


In [2]:
def build_obs(flux, flux_errs, filters, snr=10,  **extras):
    """Build a dictionary of observational data. 
    
    :param snr:
        The S/N to assign to the photometry
        
    :param ldist:
        The luminosity distance to assume for translating absolute magnitudes 
        into apparent magnitudes.
        
    :returns obs:
        A dictionary of observational data to use in the fit.
    """
    from prospect.utils.obsutils import fix_obs
    import sedpy
    obs = {}
    filternames = filters
    obs["filters"] = sedpy.observate.load_filters(filternames)
    obs["maggies"] = flux
    if len(flux_errs) != len(flux):
        obs["maggies_unc"] = (1./snr) * obs["maggies"]
    else:
        obs["maggies_unc"] = flux_errs

    obs["phot_wave"] = np.array([f.wave_effective for f in obs["filters"]])

    # We do not have a spectrum
    obs["wavelength"] = None
    obs["spectrum"] = None
    obs['unc'] = None
    obs['mask'] = None
    obs = fix_obs(obs)

    return obs


def age_of_universe_gyr_at_z(z, H0_kms_Mpc=70.0, Om=0.3, Ol=0.7):
    """
    Numerical integral to compute age of universe at redshift z (Gyr)
    using a flat LambdaCDM with H0 (km/s/Mpc), Om, Ol.
    """
    # Convert constants
    pc = 3.085677581e16       # meters
    Mpc = pc * 1e6
    H0_SI = H0_kms_Mpc * 1000.0 / Mpc   # s^-1
    sec_per_Gyr = 1e9 * 365.25 * 24 * 3600.0
    H0_Gyr = H0_SI * sec_per_Gyr        # in Gyr^-1

    # integrate from z -> zmax
    zmax = 1e5
    zs = np.linspace(z, zmax, 200000)
    integrand = 1.0 / ((1.0 + zs) * np.sqrt(Om * (1.0 + zs)**3 + Ol))
    integral = np.trapz(integrand, zs)   # in units of 1/H0
    age_gyr = integral / H0_Gyr
    return float(age_gyr)

def build_model(object_redshift=None, ldist=10.0, fixed_metallicity=None, add_duste=False, add_agn=False,
                **extras):
    """Build a prospect.models.SedModel object with optional free redshift"""
    from prospect.models.sedmodel import SedModel
    from prospect.models.templates import TemplateLibrary
    from prospect.models import priors

    # Base model
    model_params = TemplateLibrary["parametric_sfh"]
    
    # Initial guesses
    model_params["dust2"]["init"] = 1
    model_params["logzsol"]["init"] = 1.0
    model_params["mass"]["init"] = 1e8

    # default age prior bounds (will be overwritten if object_redshift provided)
    default_tage_min = 0.01   # Gyr
    default_tage_max = 13.8   # Gyr (cosmic age today)


    # Priors
    model_params["dust2"]["prior"] = priors.TopHat(mini=0.0, maxi=5.0)
    model_params["tau"]["prior"] = priors.LogUniform(mini=0.5, maxi=10.0)
    model_params["mass"]["prior"] = priors.TopHat(mini=10**6.5, maxi=10**13.5)
    model_params["logzsol"]["prior"] = priors.TopHat(mini=-2, maxi=0.3)

    # --- Metallicity ---
    if fixed_metallicity is not None:
        model_params["logzsol"]["isfree"] = True
        model_params["logzsol"]['init'] = fixed_metallicity 

    # --- Redshift ---
    if object_redshift is not None:
        model_params["zred"]["isfree"] = False
        model_params["zred"]["init"]   = object_redshift
        age_at_z = age_of_universe_gyr_at_z(object_redshift, H0_kms_Mpc=70.0, Om=0.3, Ol=0.7)
        model_params["tage"]["prior"] = priors.TopHat(mini=default_tage_min, maxi=max(default_tage_min, age_at_z))
        model_params["tage"]["init"] = min(6.0, age_at_z*0.8)
    
    else:
        # redshift is free: leave your prior or tighten if desired
        model_params["zred"]["isfree"] = True
        model_params["zred"]["init"]   = 0.7
        model_params["zred"]["prior"]  = priors.Uniform(mini=0.0, maxi=2.0)
        model_params["tage"]["prior"]  = priors.TopHat(mini=default_tage_min, maxi=default_tage_max)
        model_params["tage"]["init"]   = 4.0   # Gyr, reasonable generic guess


    # --- Dust emission ---
    if add_duste:
        model_params.update(TemplateLibrary["dust_emission"])

    # if add_duste:
    #     model_params['add_dust_emission'] = TemplateLibrary['alpha']['add_dust_emission']
    #     model_params['duste_umin'] = TemplateLibrary['alpha']['duste_umin']
    #     model_params['duste_qpah'] = TemplateLibrary['alpha']['duste_qpah']
    #     model_params['duste_gamma'] = TemplateLibrary['alpha']['duste_gamma']
    #     model_params['dust_type']['init'] = 4
    #     model_params["duste_qpah"]["prior"] = priors.TopHat(mini=0.1, maxi=10)

    if add_agn:
        model_params['fagn'] = TemplateLibrary['alpha']['fagn']
        model_params['agn_tau'] = TemplateLibrary['alpha']['agn_tau']
        model_params['add_agn_dust'] = TemplateLibrary['alpha']['add_agn_dust']
        model_params['fagn']['isfree'] = True    # make agn luminosity free, default priors
        model_params['agn_tau']['isfree'] = True
        model_params['add_agn_dust']['init'] = True

        
    return SedModel(model_params)


def build_sps(zcontinuous=1, **extras):
    """
    :param zcontinuous: 
        A vlue of 1 insures that we use interpolation between SSPs to 
        have a continuous metallicity parameter (`logzsol`)
        See python-FSPS documentation for details
    """
    from prospect.sources import CSPSpecBasis
    sps = CSPSpecBasis(zcontinuous=zcontinuous)
    return sps

def chivecfn(theta):
    """A version of lnprobfn that returns the simple uncertainty 
    normalized residual instead of the log-posterior, for use with 
    least-squares optimization methods like Levenburg-Marquardt.
    
    It's important to note that the returned chi vector does not 
    include the prior probability.
    """
    lnp_prior = model.prior_product(theta)
    if not np.isfinite(lnp_prior):
        return np.zeros(model.ndim) - np.infty

    # Generate mean model
    try:
        spec, phot, x = model.mean_model(theta, obs, sps=sps)
    except(ValueError):
        return np.zeros(model.ndim) - np.infty

    chispec = chi_spec(spec, obs)
    chiphot = chi_phot(phot, obs)
    return np.concatenate([chispec, chiphot])

In [3]:
filters = ['jwst_f090w', 'jwst_f115w', 'jwst_f150w', 'jwst_f182m', 'jwst_f200w',
            'jwst_f210m', 'jwst_f277w', 'jwst_f335m', 'jwst_f356w', 'jwst_f410m',
            'jwst_f430m', 'jwst_f444w', 'jwst_f480m']

# for i, r in enumerate(df.region):


#     flux = np.array(df.fluxes[i])/3631
#     flux_err = [] #np.array(df.errors[i])/3631
#     obs = build_obs(flux, flux_err, filters=filters, snr=10)
    
#     run_params = {}
#     run_params["snr"] = 10.0
#     wphot = obs["phot_wave"]
#     xmin, xmax = np.min(wphot)*0.8, np.max(wphot)/0.8
#     ymin, ymax = obs["maggies"].min()*0.8, obs["maggies"].max()/0.4
#     plt.rcParams["font.size"] = 18
#     figure(figsize=(16,8))
    
#     jwst_cmap = plt.get_cmap('gist_rainbow_r') 
#     jwst_norm = mcolors.Normalize(vmin=0, vmax=len(obs['filters'])-1)
#     mask = obs["phot_mask"]
#     errorbar(wphot[mask], obs['maggies'][mask], 
#              yerr=obs['maggies_unc'][mask], 
#              label='Mock photometry points',
#              marker='o', markersize=8, alpha=0.8, ls='', lw=3,
#              ecolor='black', markerfacecolor='none', markeredgecolor='black', 
#              markeredgewidth=3)
    
#     # plot Filters
#     for i, f in enumerate(obs['filters']):
#         w, t = f.wavelength.copy(), f.transmission.copy()
#         t = t / t.max()
#         t = 10**(0.2*(np.log10(ymax/ymin)))*t * ymin
#         loglog(w, t, lw=3, color=jwst_cmap(jwst_norm(i)), alpha=0.7)
    
#     xlabel('Wavelength [A]')
#     ylabel('Flux Density [maggies]')
#     xlim([xmin, xmax])
#     ylim([ymin, ymax])
#     xscale("log")
#     yscale("log")
#     legend(loc='upper left', fontsize=20, frameon=False)
#     tight_layout()
#     plt.show()

In [ ]:
filters = ['jwst_f090w', 'jwst_f115w', 'jwst_f150w', 'jwst_f182m', 'jwst_f200w',
           'jwst_f210m', 'jwst_f277w', 'jwst_f335m', 'jwst_f356w', 'jwst_f410m',
           'jwst_f430m', 'jwst_f444w', 'jwst_f480m']

jwst_cmap = plt.get_cmap('gist_rainbow_r')  # colormap
jwst_norm = mcolors.Normalize(vmin=0, vmax=len(filters)-1)

output_dir = "/beegfs/car/shenoy/JWST/SEDs/Test/"
os.makedirs(output_dir, exist_ok=True)

def process_sed(region_index, flux, flux_err=[], run_params={}):
    """
    Fit SED for a single region, save plot, and return best-fit parameters and chi2.
    """
    # converting fluxes from Jy to maggies
    flux = np.array(flux) / 3631
    flux_err = np.array(flux_err) / 3631 if flux_err else []

    # building observations based on fluxes
    obs = build_obs(flux, flux_errs=[], filters=filters, snr=10)

    # Initial SED (optional, can skip plotting if not needed)
    theta = model.theta.copy()
    initial_spec, initial_phot, initial_mfrac = model.sed(theta, obs=obs, sps=sps)

    
    a = 1.0 + model.params.get('zred', 0.0)
    wphot = obs["phot_wave"]
    wspec = obs["wavelength"] if obs["wavelength"] is not None else sps.wavelengths * a

    # fitting the model
    print(f"Fitting SED for region {region_index}...")
    output = fit_model(obs, model, sps, lnprobfn=lnprobfn, **run_params)
    print(f"Done optimization for region {region_index} in {output['optimization'][1]}s")

    # extracting best-fit parameters
    results, _ = output.get("optimization", ([], 0))
    if len(results) == 0:
        raise RuntimeError(f"Optimization failed for region {region_index}!")

    ind_best = np.argmin([r.cost for r in results])
    theta_best = results[ind_best].x.copy()

    # generating a best-fit model
    pspec, pphot, pfrac = model.mean_model(theta_best, obs=obs, sps=sps)

    # Plot SED
    xmin, xmax = np.min(wphot)*0.8, np.max(wphot)/0.8
    ymin, ymax = obs["maggies"].min()*0.8, obs["maggies"].max()/0.4

    figure(figsize=(16,8))
    plt.loglog(wspec, pspec, label='Fitted spectrum', lw=1, color='mediumpurple', alpha=1)
    plt.errorbar(
        wphot, obs['maggies'], yerr=obs['maggies_unc'],
        label='Observed photometry', marker='o', markersize=10, alpha=0.8,
        ls='', lw=3, ecolor='black', markerfacecolor='none', 
        markeredgecolor='black', markeredgewidth=3
    )

    for i, f in enumerate(obs['filters']):
        w, t = f.wavelength.copy(), f.transmission.copy()
        t = t / t.max()
        t = 10**(0.2*(np.log10(ymax/ymin)))*t * ymin
        plt.loglog(w, t, lw=3, color=jwst_cmap(jwst_norm(i)), alpha=0.7)

    plt.xlabel('Wavelength [A]')
    plt.ylabel('Flux Density [maggies]')
    plt.xlim([xmin, xmax])
    plt.ylim([ymin, ymax])
    plt.legend(loc='best', fontsize=20)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"mptesting_SEDfit_wavelet_{region_index}.png"))
    plt.close()

    return theta_best

# add df.errors[i] if available and sensible, otherwise estimated assuming S/N ratio
region_inputs = [(i, df.fluxes[i], []) for i in range(len(df))] 

if __name__ == "__main__":
    run_params = {
    "object_redshift": None,
    "fixed_metallicity": True,
    "add_duste": True,
    "add_agn": False,
    "verbose": True,
    "zcontinuous": 1, 
    "dynesty": False,
    "emcee": False,
    "optimize": True,
    "min_method": 'lm',
    "nmin": 16,
    "nwalkers": 100,
    "random_seed": 42 }
    final_vals = []
    chi2_vals = []

    np.random.seed(42)
    print("Building model...")
    model = build_model(**run_params)
    print("done")
    print("Building SPS...")
    sps = build_sps(**run_params)
    print("done")

    with Pool(processes=1) as pool:
        results = pool.starmap(process_sed, [(i, f, e, run_params) for i, f, e in region_inputs])
    

        final_vals.append(results)

    print("Finished all regions.")

Building model...
done
Building SPS...
done
Fitting SED for region 0...


/home2/shenoy/.local/lib/python3.9/site-packages/prospect/models/priors.py:125: RuntimeWarning: divide by zero encountered in log
  lnp = np.log(p)


Done optimization for region 0 in 261.07375502586365s
Fitting SED for region 1...
Done optimization for region 1 in 26.11794400215149s
Fitting SED for region 2...
Done optimization for region 2 in 23.86693263053894s
Fitting SED for region 3...
Done optimization for region 3 in 26.570120573043823s
Fitting SED for region 4...
Done optimization for region 4 in 23.3111732006073s
Fitting SED for region 5...
Done optimization for region 5 in 20.833446264266968s
Fitting SED for region 6...
Done optimization for region 6 in 58.26193833351135s
Fitting SED for region 7...
Done optimization for region 7 in 53.94314241409302s
Fitting SED for region 8...
Done optimization for region 8 in 22.988921403884888s
Fitting SED for region 9...
Done optimization for region 9 in 50.22832727432251s
Fitting SED for region 10...
Done optimization for region 10 in 20.92317247390747s
Fitting SED for region 11...
Done optimization for region 11 in 23.436800479888916s
Fitting SED for region 12...
Done optimization f